## Silver — team_crypto_silver.ticks_prices
Infers the JSON shape from the most recent bronze row using `schema_of_json` (no RDDs/UDFs - required on shared clusters). Since it reads the newest event, a field added by the producer is picked up automatically on the next run - no code change, no touching the streaming consumer.

In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema", "team_crypto_bronze")
dbutils.widgets.text("silver_schema", "team_crypto_silver")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

bronze_ticks = f"{catalog}.{bronze_schema}.ticks_stream"
silver_ticks = f"{catalog}.{silver_schema}.ticks_prices"


In [0]:
bronze_df = spark.table(bronze_ticks)

latest_value = bronze_df.orderBy(F.col("ingestion_timestamp").desc()).select("value").limit(1)
schema_ddl = latest_value.select(F.schema_of_json(F.col("value")).alias("s")).first()["s"]

df_inferred = bronze_df.select(F.from_json(F.col("value"), schema_ddl).alias("data")).select("data.*")


In [0]:
symbol_map_dict = {"bitcoin": "BTC", "ethereum": "ETH", "solana": "SOL"}
map_expr = F.create_map([F.lit(x) for pair in symbol_map_dict.items() for x in pair])

df_ticks = (df_inferred
    .withColumn("symbol", map_expr[F.col("symbol")])
    .withColumn("event_time", F.to_timestamp(F.col("ts")))
    .withColumn("source", F.lit("streaming"))
    .drop("ts")
    .dropDuplicates(["symbol", "event_time"]))


In [0]:
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

if spark.catalog.tableExists(silver_ticks):
    target = DeltaTable.forName(spark, silver_ticks)
    (target.alias("t")
        .merge(df_ticks.alias("s"), "t.symbol = s.symbol AND t.event_time = s.event_time")
        .whenNotMatchedInsertAll()
        .execute())
else:
    df_ticks.write.format("delta").saveAsTable(silver_ticks)
